# Capa Silver - Dimensión de Promociones

In [1]:
from spark_session import *
from lakehouse_utils import *
from pyspark.sql import functions as f
from pyspark.sql.window import Window

## Spark Session

In [2]:
spark = create_spark_session(
    "retail-dlh-gold",
    configs=[
        configure_iceberg_delta_extensions,
        configure_delta_catalog,
        configure_polaris_restcatalog,
        configure_retail_polaris,
        configure_s3
    ]
)

## Extracción

In [3]:
target_table = "polaris.gold.DIM_PROMOTIONS"
df_promotions = spark.table("silver.promotions")
df_dim_promotions = spark.table(target_table)

SLF4J: Failed to load class "org.slf4j.impl.StaticLoggerBinder".
SLF4J: Defaulting to no-operation (NOP) logger implementation
SLF4J: See http://www.slf4j.org/codes.html#StaticLoggerBinder for further details.


## Transformaciones

In [4]:
# Obtener valor maximo de la llave subrogada
max_sk = df_dim_promotions.select(f.coalesce(f.max(f.col("PROMO_SK")), f.lit(0))).first()[0]

# Cambiar formatos de fecha
df_promotions = df_promotions \
    .withColumn("START_DATE", f.date_format(f.col("START_DATE"), "yyyyMMdd").cast("int")) \
    .withColumnRenamed("START_DATE", "START_DK") \
    .withColumn("EXPIRY_DATE", f.date_format(f.col("EXPIRY_DATE"), "yyyyMMdd").cast("int")) \
    .withColumnRenamed("EXPIRY_DATE", "EXPIRY_DK")

# Obtener registros nuevos y calcula su llave subrogada
df_promotions_ins = df_promotions.alias("p") \
    .join(df_dim_promotions.alias("dp"), (f.col("p.PROMO_ID") == f.col("dp.PROMO_ID")), "left_anti") \
    .withColumn("PROMO_SK", max_sk + f.row_number().over(Window.orderBy(f.col("PROMO_ID")))) \
    .select(
        f.col("PROMO_SK"),
        f.col("p.*")
    )

# Obtener registros a actualizar
df_promotions_upd = df_promotions.alias("p") \
    .join(df_dim_promotions.alias("dp"), (f.col("p.PROMO_ID") == f.col("dp.PROMO_ID")), "inner") \
    .select(
        f.col("dp.PROMO_SK"),
        f.col("p.*")
    )

# Seleccion de columnas finales
df_source = df_promotions_ins.unionByName(df_promotions_upd)

## Carga

In [5]:
# Obtenemos variables para merge
join_columns = ["PROMO_SK"]

# Realiza el merge de la información
iceberg_upsert(df_source, target_table, join_columns)

## Cierre de SparkSession

In [6]:
spark.stop()